# jfinance quickstart

[jfinance](https://github.com/sgawa/jfinance) reads corporate disclosure data filed with
**EDINET**, the electronic disclosure system of the Financial Services Agency of Japan.

No registration and no API key. The API follows yfinance, so most yfinance code runs
unchanged. Coverage is annual, semi-annual and quarterly securities reports and large
shareholding reports **from fiscal 2016 onwards**.

Documentation: <https://jfnc.org/>

## Install

In [ ]:
!pip install -q jfinance

In [ ]:
import jfinance as jf
import pandas as pd

# Do not let pandas hide the middle of a long frame
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

jf.__version__

## A company

A ticker is a securities code (`7203.T`), an EDINET code (`E02144`), an ISIN
(`JP3633400001`) or an EDINET fund code (`G02925`). All four reach the same company.

In [ ]:
t = jf.Ticker("7203.T")

t.info["shortName"], t.info["edinetCode"], t.info["isin"]

### Profile

In [ ]:
pd.Series({k: t.info[k] for k in (
    "shortName", "sector", "industry", "marketSegment", "listingStatus",
    "fiscalYearEnd", "fullTimeEmployees", "sharesOutstanding",
    "capitalMillionJpy", "returnOnEquity", "equityRatio", "trailingEps",
)}, name=t.info["symbol"]).to_frame()

## Financial statements

EDINET holds the statements **as they were filed**: the same line items, in the same
order and the same hierarchy. `get_statements()` returns that. Start there, because it
is what the filing actually says — the yfinance-shaped frames come further down.

### What tables the filing contains

One annual report holds dozens of tables, not just three. `Role` is the name of the
table in the EDINET taxonomy.

In [ ]:
s = t.get_statements()

print(s.shape[0], "rows /", s["Role"].nunique(), "tables")
s["Role"].value_counts().head(12)

### One table, as filed

The helper below pulls out a single table and indents each line by its `Depth`, which
reproduces the layout of the printed statement.

Table names depend on the accounting standard. Toyota reports under IFRS, so the income
statement is `Consolidated statement of profit or loss (IFRS)`; a company on Japanese
GAAP has `Consolidated statement of income`. Use the list above to find the name.

In [ ]:
def statement(s, keyword, periods=3):
    """Pull out one table by a word in its name, indented by hierarchy.

    Rows with no amount are the headings of the statement."""
    roles = [r for r in s["Role"].unique()
             if keyword.lower() in r.lower() and not r.startswith("Notes")]
    if not roles:
        raise KeyError(f"no table matching {keyword!r}")
    d = s[s["Role"] == roles[0]]
    years = [c for c in d.columns if isinstance(c, int)][-periods:]
    out = d[years].apply(lambda col: col.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}"))
    out.index = ["    " * int(n) + str(lab) for n, lab in zip(d["Depth"], d["Label"])]
    out.index.name = roles[0]
    return out


statement(s, "profit or loss")

### Balance sheet

In [ ]:
statement(s, "financial position")

### Cash flow statement

In [ ]:
statement(s, "cash flows")

### Notes to the statements

The notes are tables too. Segment information, property, plant and equipment,
inventories — each is a `Role` beginning with `Notes -`.

In [ ]:
pd.Series([r for r in s["Role"].unique() if r.startswith("Notes -")], name="Notes")

### The same numbers in yfinance's shape

`financials`, `balance_sheet` and `cashflow` fold the filed line items into yfinance's
names, so code written for yfinance keeps working. Rows are line items, columns are
period end dates, newest first, and amounts are in yen.

Two differences from yfinance:

- **Every fiscal year is returned**, not just the last four.
- Only line items that EDINET actually discloses appear. Yahoo's own constructs
  (`Normalized EBITDA`, `Tax Effect Of Unusual Items` and the like) have no source in a
  Japanese filing, so they produce no row — the same as a company for which Yahoo has
  no data. The rows keep yfinance's order.

In [ ]:
t.financials

#### Balance sheet

In [ ]:
t.balance_sheet

#### Cash flow

In [ ]:
t.cashflow

### Periods

Quarterly reporting was abolished in April 2024 and replaced by semi-annual reporting,
so quarterly figures stop at fiscal 2023.

In [ ]:
for freq in ("yearly", "semiannual", "quarterly"):
    df = t.get_income_stmt(freq=freq)
    print(f"{freq:12} {df.shape[0]:3} rows x {df.shape[1]:2} periods   "
          f"{df.columns.min().date()} .. {df.columns.max().date()}")

### Items Yahoo Finance has no name for

Ordinary income, book value per share, headcount, and the line items particular to banks
and insurers. Only the items the company discloses are returned, so a manufacturer gets
no bank line items.

In [ ]:
t.get_jp_financials()

### Consolidation and amendments

By default: consolidated figures where they exist, with amendment filings applied.

In [ ]:
jf.set_financials_basis(consolidation="standalone", version="as_filed")
standalone = jf.Ticker("7203.T").financials

jf.set_financials_basis()          # back to the default
consolidated = jf.Ticker("7203.T").financials

pd.DataFrame({"consolidated, amended": consolidated.loc["Total Revenue"],
              "standalone, as filed": standalone.loc["Total Revenue"]})

## Shareholders

Japanese disclosure splits this across several filings, and jfinance exposes each one
separately rather than merging them.

### Top ten shareholders (annual report)

In [ ]:
t.major_shareholders

### Ownership breakdown by holder type

In [ ]:
t.major_holders

### Large shareholding reports (the 5% rule)

Filed by the holder, not the company, and covering companies and individuals as well as
institutions.

In [ ]:
h = jf.Ticker("4755.T")            # Rakuten Group
h.large_holders

Every purchase and sale the holder reported, with the price paid.

In [ ]:
h.large_holder_transactions

## Officers and pay

Officers are disclosed once a year, in the annual report. Individual remuneration is
disclosed only where consolidated pay reaches 100 million yen, so most officers never
appear in `officer_compensation`.

### The officer list

In [ ]:
t.officers

### Individual remuneration, all years

In [ ]:
t.officer_compensation

### Remuneration by category, and audit fees

In [ ]:
display(t.officer_remuneration.head(8))
t.audit_fees.head(6)

## Segments, workforce, filings

### Segment revenue

In [ ]:
seg = t.segments.query("Metric == 'seg_revenue'")
seg.pivot_table(index="Segment Label", columns="Fiscal Year",
                values="Value", aggfunc="first").tail(10)

### Workforce

In [ ]:
t.employees

### Filings

In [ ]:
# document type 120 = annual securities report
t.get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Fiscal Year", "Is Correction", "Document ID"]]

### Greenhouse gas emissions

Tagged in XBRL from the fiscal 2024 annual reports onwards, so many companies are still
empty. This is a disclosed figure, not a third-party ESG score.

In [ ]:
jf.Ticker('9432.T').emissions      # NTT

## Finding companies

### Search

In [ ]:
pd.DataFrame(jf.Search("toyota").quotes)[["symbol", "shortname", "quoteType"]]

### Industry classifications

In [ ]:
display(jf.JpSector.all())
jf.JpSector("automobiles-transportation-equipment").top_companies.head(10)

### Screener

Filters on disclosed figures, written the same way as yfinance's `screen`.

Fields derived from share prices — market capitalisation, P/E, share price — are not
available, because EDINET does not carry them.

In [ ]:
res = jf.edinet_screen(
    jf.EdinetQuery("and", [
        jf.EdinetQuery("gt", ["roe", 0.15]),
        jf.EdinetQuery("gt", ["equityRatio", 0.5]),
    ]),
    sortField="revenue",
    size=20,
)
print(res["total"], "companies match")
pd.DataFrame(res["quotes"])[
    ["symbol", "shortName", "sector", "roe", "equityRatio", "revenue"]]

#### What can be queried

In [ ]:
q = jf.EdinetQuery("gt", ["roe", 0.2])
for group, fields in q.valid_fields.items():
    print(f"{group:12} {len(fields):3}  {', '.join(list(fields)[:6])}")

## Investment trusts

25 items per reporting date. Identify a fund by its EDINET fund code (starting with
`G`), or by securities code if it is exchange traded.

In [ ]:
jf.Ticker("1306.T").get_fund_financials()

## Filings on a given day

Across all companies.

In [ ]:
jf.FilingCalendar("2026-06-25").get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Document ID"]]

## Japanese

Company and industry names come back in English by default.

Business descriptions and the names of officers and shareholders are returned in
Japanese under either setting, because EDINET holds no English original.

In [ ]:
jf.config.locale.lang = "ja-JP"

t = jf.Ticker("7203.T")
print(t.info["shortName"], "/", t.info["sector"], "/", t.info["industry"])

## What is not here

Share prices, dividend history, splits, options, analyst estimates, news, earnings
calendars and ESG scores do not exist in EDINET. Those yfinance attributes are **not
defined**, so accessing one raises `AttributeError` rather than returning something empty.

In [ ]:
for name in ("history", "dividends", "splits", "news",
             "recommendations", "earnings_dates", "sustainability"):
    print(f"{name:16} {'present' if hasattr(t, name) else 'not defined'}")

## Please read

**Amendments to filings past their EDINET public inspection period cannot be retrieved,
and may therefore not be reflected.** Older fiscal years are more likely to retain
pre-amendment values.

In [ ]:
print(jf.NOTICE_CORRECTIONS)

---

Source and terms:

```text
出典：EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）、
      PDL1.0（https://www.digital.go.jp/resources/open_data/public_data_license_v1.0）
EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）をもとに jfinance 作成
```

The same notice is in the `X-JF-Notice` header of every response.